In [15]:
import sys
from pyprojroot import here
sys.path.insert(0, str(here()))

In [16]:
import copy
import optuna
import numpy as np
from transformers import AutoTokenizer
import time

from src.train import train_single_fold, tokenize_df
from src.config import load_config
from src.data_holdout import get_pooled_df, get_fold_from_disk

import yaml
import torch
from datasets import Dataset
from pyprojroot import here
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
)

from sklearn.metrics import f1_score
from peft import LoraConfig, get_peft_model, TaskType



In [17]:
MODEL_NAME = "roberta-base"

In [12]:
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=3)
for name, module in model.named_modules():
    if "query" in name or "value" in name:
        print(name)

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 1864.40it/s, Materializing param=roberta.encoder.layer.11.output.dense.weight]              
RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.bias        | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.weight         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consid

roberta.encoder.layer.0.attention.self.query
roberta.encoder.layer.0.attention.self.value
roberta.encoder.layer.1.attention.self.query
roberta.encoder.layer.1.attention.self.value
roberta.encoder.layer.2.attention.self.query
roberta.encoder.layer.2.attention.self.value
roberta.encoder.layer.3.attention.self.query
roberta.encoder.layer.3.attention.self.value
roberta.encoder.layer.4.attention.self.query
roberta.encoder.layer.4.attention.self.value
roberta.encoder.layer.5.attention.self.query
roberta.encoder.layer.5.attention.self.value
roberta.encoder.layer.6.attention.self.query
roberta.encoder.layer.6.attention.self.value
roberta.encoder.layer.7.attention.self.query
roberta.encoder.layer.7.attention.self.value
roberta.encoder.layer.8.attention.self.query
roberta.encoder.layer.8.attention.self.value
roberta.encoder.layer.9.attention.self.query
roberta.encoder.layer.9.attention.self.value
roberta.encoder.layer.10.attention.self.query
roberta.encoder.layer.10.attention.self.value
roberta.

## Test query, value vector only vs all linear layers

### Checked to make sure had correct names for query/vector

In [13]:
# open shared settings
with open(here("configs/base.yaml"), "r") as f:
    base_data = yaml.full_load(f)

# get df
full_df = get_pooled_df()

# same pattern as collaborator: all folds
train_folds, val_folds = get_fold_from_disk(full_df)

# load tokenizer and write tokenize function
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_fold(df):
    ds = Dataset.from_pandas(df, preserve_index=False)

    def _tokenize(examples):
        return tokenizer(
            examples["abstract_conclusion"],
            examples["press_release_conclusion"],
            truncation=True,
            padding="max_length",
            max_length=base_data["max_length"],
        )

    cols_to_remove = [c for c in ds.column_names if c != "exaggeration_label"]
    ds = ds.map(_tokenize, batched=True, remove_columns=cols_to_remove)
    ds = ds.rename_column("exaggeration_label", "labels")
    ds.set_format("torch")
    return ds

# tokenize all folds
train_ds = [tokenize_fold(df) for df in train_folds]
val_ds = [tokenize_fold(df) for df in val_folds]

for i in range(len(train_ds)):
    print(f"Fold {i}: train={len(train_ds[i])} val={len(val_ds[i])}")

# define metrics function
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {"macro_f1": f1_score(labels, preds, average="macro")}


/workspace/exaggeration-detection/src/data_holdout.py:65: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .replace(ENCODED_LABELS)
Map: 100%|██████████| 106/106 [00:00<00:00, 8547.30 examples/s]

Fold 0: train=424 val=106
Fold 1: train=424 val=106
Fold 2: train=424 val=106
Fold 3: train=424 val=106
Fold 4: train=424 val=106


In [21]:


def run_quick_lora_test(target_modules, fold=0):
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=3
    )

    peft_config = LoraConfig(
        task_type=TaskType.SEQ_CLS,
        target_modules=target_modules,
        r=8,
        lora_alpha=16,
        lora_dropout=0.05,
        bias="none",
    )

    model = get_peft_model(model, peft_config)
    model.print_trainable_parameters()

    training_args = TrainingArguments(
        output_dir=str(here("results/tmp_lora_test_roberta")),
        eval_strategy="epoch",
        save_strategy="no",
        learning_rate=2e-4,
        per_device_train_batch_size=8,
        per_device_eval_batch_size=8,
        weight_decay=0.01,
        warmup_ratio=0.1,
        num_train_epochs=5,
        metric_for_best_model="macro_f1",
        fp16=True,
        seed=42,
        logging_steps=10,
        report_to="none",
        skip_memory_metrics=True,
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_ds[fold],
        eval_dataset=val_ds[fold],
        compute_metrics=compute_metrics,
    )

    start = time.time()
    trainer.train()
    metrics = trainer.evaluate()
    elapsed = time.time() - start

    del trainer, model
    torch.cuda.empty_cache()

    return {
        "target_modules": target_modules,
        "runtime_min": elapsed / 60,
        "macro_f1": metrics["eval_macro_f1"],
    }

results_qv = run_quick_lora_test(["query","value"])
results_all = run_quick_lora_test(["query", "key", "value", "dense", "out_proj"])

print(results_qv)
print(results_all)

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 1598.77it/s, Materializing param=roberta.encoder.layer.11.output.dense.weight]              
RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.bias        | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.weight         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consid

trainable params: 887,811 || all params: 125,535,750 || trainable%: 0.7072


Epoch,Training Loss,Validation Loss,Macro F1
1,0.974606,0.934494,0.253411
2,0.846590,0.936841,0.253411
3,0.916394,0.920127,0.253411
4,0.862260,0.921097,0.253411
5,1.032867,0.922612,0.253411


Loading weights: 100%|██████████| 197/197 [00:00<00:00, 2881.08it/s, Materializing param=roberta.encoder.layer.11.output.dense.weight]              
RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.bias        | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.weight         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consid

trainable params: 1,920,003 || all params: 126,567,942 || trainable%: 1.5170


Epoch,Training Loss,Validation Loss,Macro F1
1,0.924403,0.942535,0.253411
2,0.878416,0.941943,0.253411
3,0.978257,0.927976,0.253411
4,0.829582,0.958670,0.253411
5,0.980064,0.950361,0.318699


{'target_modules': ['query', 'value'], 'runtime_min': 0.4382455150286357, 'macro_f1': 0.253411306042885}
{'target_modules': ['query', 'key', 'value', 'dense', 'out_proj'], 'runtime_min': 0.6489670475323995, 'macro_f1': 0.3186991869918699}


### Ran a small test first

In [6]:
def run_quick_lora_test(target_modules, fold=0):
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=3
    )

    peft_config = LoraConfig(
        task_type=TaskType.SEQ_CLS,
        target_modules=target_modules,
        r=8,
        lora_alpha=16,
        lora_dropout=0.05,
        bias="none",
    )

    model = get_peft_model(model, peft_config)
    model.print_trainable_parameters()

    training_args = TrainingArguments(
        output_dir=str(here("results/tmp_lora_test_roberta")),
        eval_strategy="epoch",
        save_strategy="no",
        learning_rate=2e-4,
        per_device_train_batch_size=8,
        per_device_eval_batch_size=8,
        weight_decay=0.01,
        warmup_ratio=0.1,
        num_train_epochs=5,
        metric_for_best_model="macro_f1",
        fp16=True,
        seed=42,
        logging_steps=10,
        report_to="none",
        skip_memory_metrics=True,
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_ds[fold],
        eval_dataset=val_ds[fold],
        compute_metrics=compute_metrics,
    )

    start = time.time()
    trainer.train()
    metrics = trainer.evaluate()
    elapsed = time.time() - start

    del trainer, model
    torch.cuda.empty_cache()

    return {
        "target_modules": target_modules,
        "runtime_min": elapsed / 60,
        "macro_f1": metrics["eval_macro_f1"],
    }

results_qv = run_quick_lora_test(["query","value"])
results_all = run_quick_lora_test("all-linear")

print(results_qv)
print(results_all)

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 1428.80it/s, Materializing param=roberta.encoder.layer.11.output.dense.weight]              
RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.bias        | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.weight         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consid

trainable params: 887,811 || all params: 125,535,750 || trainable%: 0.7072


Epoch,Training Loss,Validation Loss,Macro F1
1,0.983513,0.931947,0.253411
2,0.850388,0.937608,0.253411
3,0.913065,0.920926,0.253411
4,0.865106,0.921930,0.253411
5,1.027179,0.924406,0.253411


Loading weights: 100%|██████████| 197/197 [00:00<00:00, 1574.35it/s, Materializing param=roberta.encoder.layer.11.output.dense.weight]              
RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.bias        | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.weight         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consid

trainable params: 1,920,003 || all params: 126,567,942 || trainable%: 1.5170


Epoch,Training Loss,Validation Loss,Macro F1
1,0.922802,0.940363,0.253411
2,0.876832,0.944502,0.253411
3,0.981848,0.930095,0.253411
4,0.838548,0.947200,0.253411
5,0.987828,0.946294,0.328457


{'target_modules': ['query', 'value'], 'runtime_min': 0.6380618373552959, 'macro_f1': 0.253411306042885}
{'target_modules': 'all-linear', 'runtime_min': 1.073991020520528, 'macro_f1': 0.3284566838783706}


### Then ran a bigger test on all 5 folds

In [14]:
def run_lora_5fold_test(target_modules, label):
    fold_scores = []
    fold_times = []

    for fold in range(5):
        model = AutoModelForSequenceClassification.from_pretrained(
            MODEL_NAME,
            num_labels=3
        )

        peft_config = LoraConfig(
            task_type=TaskType.SEQ_CLS,
            target_modules=target_modules,
            r=8,
            lora_alpha=16,
            lora_dropout=0.05,
            bias="none",
        )

        model = get_peft_model(model, peft_config)

        if fold == 0:
            print(f"\n=== {label} ===")
            model.print_trainable_parameters()

        training_args = TrainingArguments(
            output_dir=str(here(f"results/tmp_lora_compare_{label}/fold-{fold}")),
            eval_strategy="epoch",
            save_strategy="no",
            learning_rate=2e-4,
            per_device_train_batch_size=8,
            per_device_eval_batch_size=8,
            weight_decay=0.01,
            warmup_ratio=0.1,
            num_train_epochs=5, 
            fp16=True,
            seed=42,
            logging_steps=10,
            report_to="none",
            skip_memory_metrics=True,
        )

        trainer = Trainer(
            model=model,
            args=training_args,
            train_dataset=train_ds[fold],
            eval_dataset=val_ds[fold],
            compute_metrics=compute_metrics,
        )

        start = time.time()
        trainer.train()
        metrics = trainer.evaluate()
        elapsed = time.time() - start

        fold_scores.append(metrics["eval_macro_f1"])
        fold_times.append(elapsed / 60)

        print(
            f"{label} | fold {fold} | "
            f"macro_f1={metrics['eval_macro_f1']:.4f} | "
            f"runtime_min={elapsed/60:.2f}"
        )

        del trainer, model
        torch.cuda.empty_cache()

    return {
        "label": label,
        "target_modules": target_modules,
        "fold_scores": fold_scores,
        "mean_macro_f1": float(np.mean(fold_scores)),
        "std_macro_f1": float(np.std(fold_scores)),
        "fold_times_min": fold_times,
        "mean_runtime_min": float(np.mean(fold_times)),
        "total_runtime_min": float(np.sum(fold_times)),
    }

results_qv = run_lora_5fold_test(["query", "value"], "qv")
results_all = run_lora_5fold_test("all-linear", "all_linear")

print("\nQ/V results:")
print(results_qv)

print("\nAll-linear results:")
print(results_all)

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 1319.95it/s, Materializing param=roberta.encoder.layer.11.output.dense.weight]              
RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.bias        | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.weight         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consid


=== qv ===
trainable params: 887,811 || all params: 125,535,750 || trainable%: 0.7072


Epoch,Training Loss,Validation Loss,Macro F1
1,0.984033,0.936017,0.253411
2,0.853155,0.936788,0.253411
3,0.914195,0.923126,0.253411
4,0.862415,0.925424,0.253411
5,1.005362,0.930286,0.253411


qv | fold 0 | macro_f1=0.2534 | runtime_min=0.65


Loading weights: 100%|██████████| 197/197 [00:00<00:00, 1413.91it/s, Materializing param=roberta.encoder.layer.11.output.dense.weight]              
RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.bias        | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.weight         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consid

...

Epoch,Training Loss,Validation Loss,Macro F1
1,0.904086,0.934681,0.253411
2,1.102347,0.940204,0.253411
3,0.941922,0.926599,0.253411
4,0.790726,0.953403,0.253411
5,0.990363,0.946504,0.253411


all_linear | fold 3 | macro_f1=0.2534 | runtime_min=1.12


Loading weights: 100%|██████████| 197/197 [00:00<00:00, 1333.91it/s, Materializing param=roberta.encoder.layer.11.output.dense.weight]              
RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.bias        | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.weight         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consid

Epoch,Training Loss,Validation Loss,Macro F1
1,1.089485,0.954585,0.253411
2,0.873517,0.927838,0.253411
3,0.929337,0.924084,0.253411
4,0.918697,0.910598,0.253411
5,0.915736,0.903039,0.253411


all_linear | fold 4 | macro_f1=0.2534 | runtime_min=1.01

Q/V results:
{'label': 'qv', 'target_modules': ['query', 'value'], 'fold_scores': [0.253411306042885, 0.253411306042885, 0.253411306042885, 0.253411306042885, 0.253411306042885], 'mean_macro_f1': 0.253411306042885, 'std_macro_f1': 0.0, 'fold_times_min': [0.6537256518999736, 0.6718911528587341, 0.5995204528172811, 0.6142852505048116, 0.668046506245931], 'mean_runtime_min': 0.6414938028653463, 'total_runtime_min': 3.2074690143267315}

All-linear results:
{'label': 'all_linear', 'target_modules': 'all-linear', 'fold_scores': [0.3186991869918699, 0.30487173800547057, 0.30921855921855923, 0.253411306042885, 0.253411306042885], 'mean_macro_f1': 0.2879224192603339, 'std_macro_f1': 0.02853085447742718, 'fold_times_min': [1.0799602468808491, 1.0006040970484416, 1.1068783442179362, 1.1215750654538472, 1.0120890974998473], 'mean_runtime_min': 1.0642213702201844, 'total_runtime_min': 5.321106851100922}


### With more training epochs, using all-linear was clear winner

## Full Optuna Experiment

In [26]:

# open shared settings
with open(here("configs/base.yaml"), "r") as f:
    base_data = yaml.full_load(f)

# get df
full_df = get_pooled_df()

# same pattern as collaborator: all folds
train_folds, val_folds = get_fold_from_disk(full_df)

# load tokenizer and write tokenize function
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_fold(df):
    ds = Dataset.from_pandas(df, preserve_index=False)

    def _tokenize(examples):
        return tokenizer(
            examples["abstract_conclusion"],
            examples["press_release_conclusion"],
            truncation=True,
            padding="max_length",
            max_length=base_data["max_length"],
        )

    cols_to_remove = [c for c in ds.column_names if c != "exaggeration_label"]
    ds = ds.map(_tokenize, batched=True, remove_columns=cols_to_remove)
    ds = ds.rename_column("exaggeration_label", "labels")
    ds.set_format("torch")
    return ds

# tokenize all folds
train_ds = [tokenize_fold(df) for df in train_folds]
val_ds = [tokenize_fold(df) for df in val_folds]

for i in range(len(train_ds)):
    print(f"Fold {i}: train={len(train_ds[i])} val={len(val_ds[i])}")

# define metrics function
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {"macro_f1": f1_score(labels, preds, average="macro")}

# def build_lora_model():
#     model = AutoModelForSequenceClassification.from_pretrained(
#         MODEL_NAME,
#         num_labels=3
#     )

#     # LoRA applied broadly to linear layers
#     peft_config = LoraConfig(
#         task_type=TaskType.SEQ_CLS,
#         target_modules="all-linear",
#         r=16,                # placeholder, overwritten per trial below
#         lora_alpha=32,       # placeholder, overwritten per trial below
#         lora_dropout=0.05,   # placeholder, overwritten per trial below
#         bias="none",
#     )

#     model = get_peft_model(model, peft_config)
#     return model

# set up Optuna objective
def objective(trial):
    folds_to_run = [0, 1, 3] # folds chosen based on previous experiments

    # LoRA-specific search space
    lr = trial.suggest_float("learning_rate", 1e-5, 5e-4, log=True)
    batch_size = trial.suggest_categorical("per_device_train_batch_size", [8])
    weight_decay = trial.suggest_float("weight_decay", 0.0, 0.1)
    warmup_ratio = trial.suggest_float("warmup_ratio", 0.05, 0.2)
    num_epochs = 8

    lora_r = trial.suggest_categorical("lora_r", [2, 4, 8, 16])
    lora_alpha = trial.suggest_categorical("lora_alpha", [4, 8, 16, 32])
    lora_dropout = trial.suggest_categorical("lora_dropout", [0.0, 0.05, 0.1])

    f1_scores = []
    fold_times_min = []

    for fold in folds_to_run:
        fold_start = time.time()
        model = AutoModelForSequenceClassification.from_pretrained(
            MODEL_NAME,
            num_labels=3
        )

        peft_config = LoraConfig(
            task_type=TaskType.SEQ_CLS,
            target_modules=["query", "key", "value", "dense", "out_proj"], #approximation of all linear, get same results  in testing
            r=lora_r,
            lora_alpha=lora_alpha,
            lora_dropout=lora_dropout,
            bias="none",
        )

        model = get_peft_model(model, peft_config)

        # confirm correct number of trainable parameters
        if fold == 0:
            model.print_trainable_parameters()

        training_args = TrainingArguments(
            output_dir=str(here(f"results/optuna/roberta_lora/trial-{trial.number}/fold-{fold}")),
            eval_strategy=base_data["eval_strategy"],
            save_strategy="no",              # no checkpoints this time
            learning_rate=lr,
            per_device_train_batch_size=batch_size,
            per_device_eval_batch_size=batch_size,
            weight_decay=weight_decay,
            warmup_ratio=warmup_ratio,
            num_train_epochs=num_epochs,
            metric_for_best_model="macro_f1",
            fp16=True,
            seed=base_data["seed"],
            logging_steps=10,
            report_to=base_data["report_to"],
            skip_memory_metrics=True,
        )

        trainer = Trainer(
            model=model,
            args=training_args,
            train_dataset=train_ds[fold],
            eval_dataset=val_ds[fold],
            compute_metrics=compute_metrics,
            callbacks=[EarlyStoppingCallback(early_stopping_patience=10)],
        )

        trainer.train()
        eval_metrics = trainer.evaluate()
        f1_scores.append(eval_metrics["eval_macro_f1"])


        fold_time_min = (time.time() - fold_start) / 60
        fold_times_min.append(fold_time_min)
        print(
            f"Trial {trial.number} | Fold {fold} | "
            f"F1={eval_metrics['eval_macro_f1']:.4f} | "
            f"Time={fold_time_min:.2f} min"
        )

        del trainer
        del model
        torch.cuda.empty_cache()

    return float(np.mean(f1_scores))

# study definition
db_path = here("results/optuna/optuna_study_roberta_lora.db")
storage_url = f"sqlite:///{db_path}"
n_startup_trials = 5

print(storage_url)

study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(
        n_startup_trials=n_startup_trials,
        seed=42,
    ),
    study_name="roberta-lora-hyperparameter-search",
    storage=storage_url,
    load_if_exists=True
)

study.optimize(objective, n_trials=10) #adding 10 more trials because did 40 before

# best trials
print(f"\n{'='*60}")
print(f"  Best macro_f1: {study.best_trial.value:.4f}")
print(f"  Best params:   {study.best_trial.params}")
print(f"{'='*60}")

/workspace/exaggeration-detection/src/data_holdout.py:65: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .replace(ENCODED_LABELS)
Map: 100%|██████████| 106/106 [00:00<00:00, 11836.96 examples/s]
[I 2026-03-23 20:27:41,351] Using an existing study with name 'roberta-lora-hyperparameter-search' instead of creating a new one.


Fold 0: train=424 val=106
Fold 1: train=424 val=106
Fold 2: train=424 val=106
Fold 3: train=424 val=106
Fold 4: train=424 val=106
sqlite:////workspace/exaggeration-detection/results/optuna/optuna_study_roberta_lora.db


Loading weights: 100%|██████████| 197/197 [00:00<00:00, 1361.53it/s, Materializing param=roberta.encoder.layer.11.output.dense.weight]              
RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.bias        | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.weight         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consid

trainable params: 924,675 || all params: 125,572,614 || trainable%: 0.7364


Using EarlyStoppingCallback without load_best_model_at_end=True. Once training is finished, the best model will not be loaded automatically.


...

Trial 54 | Fold 1 | F1=0.2534 | Time=0.62 min


Loading weights: 100%|██████████| 197/197 [00:00<00:00, 2356.15it/s, Materializing param=roberta.encoder.layer.11.output.dense.weight]              
RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.bias        | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.weight         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consid

Epoch,Training Loss,Validation Loss,Macro F1
1,0.941168,0.935353,0.253411
2,1.159681,0.954530,0.253411
3,0.980606,0.957142,0.253411
4,0.869675,0.963430,0.253411
5,1.037553,0.933152,0.253411
6,0.999973,0.942406,0.253411
7,0.838803,0.937055,0.253411
8,1.048283,0.933799,0.253411


[I 2026-03-23 20:46:44,091] Trial 54 finished with value: 0.3250187699689417 and parameters: {'learning_rate': 0.0004145312748290846, 'per_device_train_batch_size': 8, 'weight_decay': 0.07353548817687415, 'warmup_ratio': 0.12778648288561972, 'lora_r': 2, 'lora_alpha': 16, 'lora_dropout': 0.0}. Best is trial 20 with value: 0.4949775285111037.


Trial 54 | Fold 3 | F1=0.2534 | Time=0.62 min

  Best macro_f1: 0.4950
  Best params:   {'learning_rate': 0.0004967831504102713, 'per_device_train_batch_size': 8, 'weight_decay': 0.04558767968874666, 'warmup_ratio': 0.12016661619597607, 'lora_r': 2, 'lora_alpha': 16, 'lora_dropout': 0.05}


In [20]:
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=3)

linear_names = []
for name, module in model.named_modules():
    if module.__class__.__name__ == "Linear":
        linear_names.append(name)

for n in linear_names:
    print(n)

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 1408.49it/s, Materializing param=roberta.encoder.layer.11.output.dense.weight]              
RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.bias        | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.weight         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consid

roberta.encoder.layer.0.attention.self.query
roberta.encoder.layer.0.attention.self.key
roberta.encoder.layer.0.attention.self.value
roberta.encoder.layer.0.attention.output.dense
roberta.encoder.layer.0.intermediate.dense
roberta.encoder.layer.0.output.dense
roberta.encoder.layer.1.attention.self.query
roberta.encoder.layer.1.attention.self.key
roberta.encoder.layer.1.attention.self.value
roberta.encoder.layer.1.attention.output.dense
roberta.encoder.layer.1.intermediate.dense
roberta.encoder.layer.1.output.dense
roberta.encoder.layer.2.attention.self.query
roberta.encoder.layer.2.attention.self.key
roberta.encoder.layer.2.attention.self.value
roberta.encoder.layer.2.attention.output.dense
roberta.encoder.layer.2.intermediate.dense
roberta.encoder.layer.2.output.dense
roberta.encoder.layer.3.attention.self.query
roberta.encoder.layer.3.attention.self.key
roberta.encoder.layer.3.attention.self.value
roberta.encoder.layer.3.attention.output.dense
roberta.encoder.layer.3.intermediate.den

In [27]:

print(study.best_trial)
print(study.best_value)
print(study.best_params)

FrozenTrial(number=20, state=<TrialState.COMPLETE: 1>, values=[0.4949775285111037], datetime_start=datetime.datetime(2026, 3, 23, 19, 28, 32, 703837), datetime_complete=datetime.datetime(2026, 3, 23, 19, 30, 37, 468875), params={'learning_rate': 0.0004967831504102713, 'per_device_train_batch_size': 8, 'weight_decay': 0.04558767968874666, 'warmup_ratio': 0.12016661619597607, 'lora_r': 2, 'lora_alpha': 16, 'lora_dropout': 0.05}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'learning_rate': FloatDistribution(high=0.0005, log=True, low=1e-05, step=None), 'per_device_train_batch_size': CategoricalDistribution(choices=(8,)), 'weight_decay': FloatDistribution(high=0.1, log=False, low=0.0, step=None), 'warmup_ratio': FloatDistribution(high=0.2, log=False, low=0.05, step=None), 'lora_r': CategoricalDistribution(choices=(2, 4, 8, 16)), 'lora_alpha': CategoricalDistribution(choices=(4, 8, 16, 32)), 'lora_dropout': CategoricalDistribution(choices=(0.0, 0.05, 0.1))}, trial